In [1]:
import pandas as pd

In [ ]:
# 1. 고객 데이터에서 나이를 연령대 분리
cust = pd.read_csv("../data/customers.csv")

In [ ]:
cust.info()

# 1. birth_date 를 drop4696. 나머지 drop. 중요한 데이터니까 birth_date가 NaN인 데이터는 제거
# 2. birth_date의 Dtype: str. datetime으로 변경 -> 나이 -> cut()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  5000 non-null   int64
 1   name         5000 non-null   str  
 2   gender       4696 non-null   str  
 3   birth_date   4616 non-null   str  
 4   signup_date  5000 non-null   str  
 5   city         5000 non-null   str  
 6   email        4752 non-null   str  
dtypes: int64(1), str(6)
memory usage: 544.3 KB


In [6]:
# 1.1 NaN 제거
# cust.dropna(subset=["birth_date"])  # 복사본으로 작업한 것. cust에 대입해야.
cust = cust.dropna(subset=["birth_date"])
cust.info()

<class 'pandas.DataFrame'>
Index: 4616 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  4616 non-null   int64
 1   name         4616 non-null   str  
 2   gender       4327 non-null   str  
 3   birth_date   4616 non-null   str  
 4   signup_date  4616 non-null   str  
 5   city         4616 non-null   str  
 6   email        4383 non-null   str  
dtypes: int64(1), str(6)
memory usage: 543.5 KB


In [5]:
cust.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  5000 non-null   int64
 1   name         5000 non-null   str  
 2   gender       4696 non-null   str  
 3   birth_date   4616 non-null   str  
 4   signup_date  5000 non-null   str  
 5   city         5000 non-null   str  
 6   email        4752 non-null   str  
dtypes: int64(1), str(6)
memory usage: 544.3 KB


In [8]:
# 1.2 birth_date 데이터 타입 현재 "str"이다. 데이터타입을 datetime64로 변경
cust["birth_date"] = pd.to_datetime(cust["birth_date"])
cust.info()


<class 'pandas.DataFrame'>
Index: 4616 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  4616 non-null   int64         
 1   name         4616 non-null   str           
 2   gender       4327 non-null   str           
 3   birth_date   4616 non-null   datetime64[us]
 4   signup_date  4616 non-null   str           
 5   city         4616 non-null   str           
 6   email        4383 non-null   str           
dtypes: datetime64[us](1), int64(1), str(5)
memory usage: 497.9 KB


In [9]:
# 1.3 나이 구하기 : 현재 연도(2026) - 생년
cust["age"] = 2026 - cust["birth_date"].dt.year
cust.head()

,customer_id,name,gender,birth_date,signup_date,city,email,age
0,3293,권준서,여,1954-04-29,2020-04-26,부산,user3293@example.com,72
1,1862,박준선,F,1980-05-19,2023-09-01,서울,user1862@example.com,46
2,4955,전영경,여,1980-06-08,2024-04-30,인천,user4955@example.com,46
3,4653,한재하,M,1971-06-09,2023-08-06,서울,user4653@example.com,55
4,3331,송준연,여,1971-02-01,2023-12-20,대전,user3331@example.com,55


In [14]:
# 1.4 나이로 10대, 20대, 30대, 40대, 50대, 60대 이상 구간화
bins = [0,19,29,39,49,59,200]
labels = ["10대 이하","20대","30대","40대","50대","60대 이상"]
cust["age_group"] = pd.cut(cust["age"], bins=bins, labels=labels, include_lowest=True)  
# 0살은 빠짐. so, include_lowest=True로 포함.

In [15]:
cust["age_group"].value_counts().sort_index()

age_group
10대 이하       0
20대        818
30대        887
40대        894
50대        889
60대 이상    1128
Name: count, dtype: int64

In [ ]:
# 2. 가격을 균등 분할 : qcut
# 데이터 로딩 : 상품 데이터
prod = pd.read_csv("../data/products.csv")
prod.head()
# price = 0 이 있음.

,product_id,product_name,category,price,cost
0,119,플러스 홍차,식품,0,3700.0
1,95,프리미엄 노트북,전자,68600.0,28900.0
2,112,프리미엄 원두커피,식품,10500.0,4700.0
3,164,베이직 마우스,전자,52100.0,32600.0
4,274,베이직 홍차,식품,16000.0,10500.0


In [ ]:
prod.info()
# Dtype = str ("원",","값이 있음.)

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    500 non-null    int64  
 1   product_name  500 non-null    str    
 2   category      500 non-null    str    
 3   price         500 non-null    str    
 4   cost          500 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.5 KB


In [ ]:
# 가격 데이터에 대한 정제
# 2.1 원, , => str, 숫자 변경
price = (
    prod["price"].astype(str)
    .str.replace("원","", regex=False)
    .str.replace(",","", regex=False)
)


pandas.Series

In [19]:
# 2.2 숫자 변경 : pd.to_numeric(Series,errors...)
prod["price"] = pd.to_numeric(price, errors='coerce')

In [21]:
# 2.3 가격이 0보다 큰 행 추출 : 0이거나 작은 행 삭제.
prod = prod[prod["price"]>0]
prod.head()

,product_id,product_name,category,price,cost
1,95,프리미엄 노트북,전자,68600.0,28900.0
2,112,프리미엄 원두커피,식품,10500.0,4700.0
3,164,베이직 마우스,전자,52100.0,32600.0
4,274,베이직 홍차,식품,16000.0,10500.0
5,55,프리미엄 니트,의류,3700.0,1900.0


In [ ]:
# 분위 수 기준 4등급(같은 개수씩으로 분할)
prod["price_tier"] = pd.qcut(
    # x, q, label이 핵심.
    prod["price"],  # 분할할 데이터 지정
    q=4,            # 분할할 개수
    labels=["저가","중가","고가","최고가"]
)

count     492
unique      4
top        저가
freq      124
Name: price_tier, dtype: object

In [25]:
prod["price_tier"].value_counts()

price_tier
저가     124
중가     123
최고가    123
고가     122
Name: count, dtype: int64

In [ ]:
# 3. orders.csv => status : 읽기 쉬운 라벨로 매핑
# 예 : "canceled" => "취소"
# 주문 데이터 로딩
# 
orders = pd.read_csv("../data/orders.csv")

In [27]:
orders["status"].unique()

<ArrowStringArray>
['delivered', 'canceled', 'paid', 'shipped', 'returned']
Length: 5, dtype: str

In [28]:
smap = {
    "paid":"결제완료",
    "delivered":"배송완료",
    "shipped":"배송중",
    "canceled":"취소",
    "returned":"반품"
}
orders["status_ko"] = orders["status"].map(smap)
orders["status_ko"].value_counts()

status_ko
배송완료    109976
배송중      30017
결제완료     29993
취소       20039
반품        9975
Name: count, dtype: int64